In [5]:
"""
Self-contained error mitigation figure generation.
Input : error_mitigation_results.json
Output: fig_em1_bar_comparison.pdf
        fig_em2_zne_extrapolation.pdf
        fig_em3_bootstrap_overlay.pdf
        fig_em_panel.pdf
"""

import json, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# =============================================================================
#  CONFIG
# =============================================================================
EM_JSON = "/Users/nandan/Desktop/CTCs/IBM/error_mitigation_results.json"
OUT_DIR = "./"

# =============================================================================
#  Load data
# =============================================================================
with open(EM_JSON) as f:
    D = json.load(f)

res   = D['results']
meta  = D['metadata']
shots = meta['shots']

# Condition display config: (key, label, colour, hatch)
CONDITIONS = [
    ('unmitigated', 'Unmitigated',              '#7F8C8D', ''),
    ('dd',          'DD (XX)',                   '#2980B9', ''),
    ('readout',     'Readout Mit.\n(mthree)',     '#8E44AD', '///'),
    ('zne',         'ZNE\n($\\lambda$=1,3,5)',    '#C0392B', ''),
]
C_IDEAL = '#27AE60'
os.makedirs(OUT_DIR, exist_ok=True)

# Style
plt.rcParams.update({
    'font.family': 'DejaVu Serif', 'font.size': 11,
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'legend.fontsize': 9.5, 'figure.dpi': 180,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.8, 'pdf.fonttype': 42,
})

# =============================================================================
#  FIG 1 — Bar chart: F and p_succ side-by-side for all conditions
# =============================================================================
def fig_bar():
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2))

    keys    = [k for k,_,_,_ in CONDITIONS if res.get(k) is not None]
    labels  = [l for k,l,_,_ in CONDITIONS if res.get(k) is not None]
    colors  = [c for k,_,c,_ in CONDITIONS if res.get(k) is not None]
    hatches = [h for k,_,_,h in CONDITIONS if res.get(k) is not None]

    F_vals = [res[k]['F']      for k in keys]
    p_vals = [res[k]['p_succ'] for k in keys]
    F_elo  = [res[k]['F']    - res[k]['F_lo']  for k in keys]
    F_ehi  = [res[k]['F_hi'] - res[k]['F']     for k in keys]

    x = np.arange(len(keys))

    # ── (a) Fidelity ──────────────────────────────────────────────────────────
    ax = axes[0]
    bars = ax.bar(x, F_vals, color=colors, width=0.55,
                  yerr=[F_elo, F_ehi],
                  error_kw=dict(elinewidth=1.6, capsize=6,
                                capthick=1.6, ecolor='#222'),
                  zorder=3, edgecolor='white')
    for bar, h in zip(bars, hatches):
        bar.set_hatch(h)

    ax.axhline(1.0, color=C_IDEAL, ls='--', lw=1.2, alpha=0.6,
               label='Ideal $F=1$')
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9.5)
    ymin = max(0.5, min(F_vals) - 0.06)
    ax.set_ylim(ymin, 1.06)
    ax.set_ylabel('$F(\\rho_Y,\\, \\rho_M)$')
    ax.set_title('Output fidelity', pad=8)
    ax.yaxis.grid(True, alpha=0.3, zorder=0)
    ax.legend(fontsize=9, framealpha=0.9)

    for xi, (fv, fl, fh) in enumerate(zip(F_vals, F_elo, F_ehi)):
        ax.text(xi, fv + fh + 0.005, f'{fv:.4f}',
                ha='center', fontsize=8.5, color='#111')

    # Improvement arrows from unmitigated
    um_F = res['unmitigated']['F']
    for xi, (k, fv) in enumerate(zip(keys[1:], F_vals[1:]), start=1):
        delta = fv - um_F
        if abs(delta) > 0.001:
            sign = '+' if delta >= 0 else ''
            col  = 'white'
            ax.text(xi, ymin + 0.01, f'{sign}{delta:.4f}',
                    ha='center', fontsize=8, color=col, style='italic')

    # ── (b) p_succ ────────────────────────────────────────────────────────────
    ax2 = axes[1]
    p_elo = [res[k]['p_succ'] - res[k].get('p_lo', res[k]['p_succ']-0.01)
             for k in keys]
    p_ehi = [res[k].get('p_hi', res[k]['p_succ']+0.01) - res[k]['p_succ']
             for k in keys]
    bars2 = ax2.bar(x, p_vals, color=colors, width=0.55,
                    yerr=[p_elo, p_ehi],
                    error_kw=dict(elinewidth=1.6, capsize=6,
                                  capthick=1.6, ecolor='#222'),
                    zorder=3, edgecolor='white')
    for bar, h in zip(bars2, hatches):
        bar.set_hatch(h)
    ax2.axhline(0.25, color=C_IDEAL, ls='--', lw=1.2, alpha=0.6,
                label='Ideal $p_{\\rm succ}=0.25$')
    ax2.set_xticks(x); ax2.set_xticklabels(labels, fontsize=9.5)
    ax2.set_ylim(0, 0.34); ax2.set_ylabel('$p_{\\rm succ}$')
    ax2.set_title('Post-selection rate', pad=8)
    ax2.yaxis.grid(True, alpha=0.3, zorder=0)
    ax2.legend(fontsize=9, framealpha=0.9)
    for xi, pv in enumerate(p_vals):
        ax2.text(xi, pv + 0.013, f'{pv:.4f}',
                 ha='center', fontsize=8.5, color='#111')

    fig.suptitle(
        f'Error mitigation comparison — ibm_torino  '
        f'({shots:,} shots/basis,  95% bootstrap CI on $F$)',
        fontsize=11, y=1.01
    )
    fig.tight_layout()
    return fig

# =============================================================================
#  FIG 2 — ZNE extrapolation curve
# =============================================================================
def fig_zne():
    fig, ax = plt.subplots(figsize=(5.8, 4.2))
    zne = res['zne']
    lambdas = zne['lambdas']
    F_vals  = [zne['F_at_lambda'][str(l)] for l in lambdas]
    p_vals  = [zne['p_at_lambda'][str(l)] for l in lambdas]

    # Data points
    ax.scatter(lambdas, F_vals, color='#C0392B', s=70, zorder=5,
               label='Measured $F(\\lambda)$')

    # Extrapolation line
    lam_ext = np.linspace(0, max(lambdas)*1.05, 200)
    coeffs  = zne['fit_coeffs']
    F_fit   = np.polyval(coeffs, lam_ext)
    ax.plot(lam_ext, F_fit, color='#C0392B', lw=2.0, ls='--',
            label=f'Linear fit  ($F_{{\\rm ZNE}}={zne["F"]:.4f}$)')

    # Extrapolated point at λ=0
    ax.scatter([0], [zne['F']], color='#C0392B', s=120, marker='*',
               zorder=6, label=f'$F_{{\\rm ZNE}}(\\lambda=0)={zne["F"]:.4f}$')

    # Unmitigated reference
    ax.axhline(res['unmitigated']['F'], color='#7F8C8D',
               ls=':', lw=1.4, alpha=0.8,
               label=f'Unmitigated $F={res["unmitigated"]["F"]:.4f}$')

    # ZNE CI band
    ax.axhspan(zne['F_lo'], zne['F_hi'],
               alpha=0.15, color='#C0392B',
               label=f'ZNE 95% CI [{zne["F_lo"]:.4f},{zne["F_hi"]:.4f}]')

    # Ideal
    ax.axhline(1.0, color=C_IDEAL, ls='-.', lw=1.2, alpha=0.6,
               label='Ideal $F=1$')

    # Mark λ=0 on x-axis
    ax.axvline(0, color='#999', lw=0.8, ls=':')
    ax.text(0.1, ax.get_ylim()[0]+0.01, '$\\lambda=0$\n(ideal noise)',
            fontsize=8.5, color='#555')

    ax.set_xlabel('Noise amplification factor $\\lambda$')
    ax.set_ylabel('Fidelity $F(\\rho_Y,\\, \\rho_M)$')
    ax.set_title('ZNE extrapolation — digital gate folding\n'
                 f'ibm\\_torino  ·  $\\lambda \\in ${{{lambdas}}}  ·  '
                 f'linear fit to $F(0)$', pad=8)
    ax.legend(framealpha=0.9, edgecolor='#ccc', fontsize=9, loc='upper right')
    ax.set_xlim(-0.3, max(lambdas)+0.5)
    ax.yaxis.grid(True, alpha=0.3, zorder=0)
    fig.tight_layout()
    return fig

# =============================================================================
#  FIG 3 — Bootstrap F distributions overlaid for all conditions
# =============================================================================
def fig_bootstrap():
    fig, ax = plt.subplots(figsize=(6.5, 4.2))

    for key, label, color, _ in CONDITIONS:
        r = res.get(key)
        if r is None: continue
        bsF = np.array(r['bootstrap_F'])
        med = np.median(bsF)
        lo  = np.percentile(bsF, 2.5)
        hi  = np.percentile(bsF, 97.5)
        ax.hist(bsF, bins=45, color=color, alpha=0.60,
                edgecolor='white', lw=0.4, zorder=3,
                label=f'{label}  ($F={r["F"]:.4f}$)')
        ax.axvline(med, color=color, lw=1.8, zorder=4)
        ax.axvline(lo,  color=color, lw=1.0, ls='--', zorder=4)
        ax.axvline(hi,  color=color, lw=1.0, ls='--', zorder=4)

    ax.axvline(1.0, color=C_IDEAL, lw=1.6, ls=':', zorder=5,
               label='Ideal $F=1$')
    ax.set_xlabel('Fidelity $F(\\rho_Y,\\, \\rho_M)$')
    ax.set_ylabel('Bootstrap count')
    ax.set_title('Bootstrap $F$ distributions — all mitigation conditions\n'
                 '(solid = median, dashed = 95% CI boundaries)', pad=8)
    ax.legend(framealpha=0.9, edgecolor='#ccc', fontsize=8.5,
              loc='upper left')
    ax.yaxis.grid(True, alpha=0.3, zorder=0)
    fig.tight_layout()
    return fig

# =============================================================================
#  FIG 4 — Combined panel
# =============================================================================
def fig_panel():
    fig = plt.figure(figsize=(15.0, 5.0))
    gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.36)

    keys    = [k for k,_,_,_ in CONDITIONS if res.get(k) is not None]
    labels  = [l for k,l,_,_ in CONDITIONS if res.get(k) is not None]
    colors  = [c for k,_,c,_ in CONDITIONS if res.get(k) is not None]
    hatches = [h for k,_,_,h in CONDITIONS if res.get(k) is not None]
    x = np.arange(len(keys))

    # ── (a) Fidelity bar ─────────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0])
    F_vals = [res[k]['F']    for k in keys]
    F_elo  = [res[k]['F']    - res[k]['F_lo'] for k in keys]
    F_ehi  = [res[k]['F_hi'] - res[k]['F']    for k in keys]
    bars=ax1.bar(x,F_vals,color=colors,width=0.55,
                 yerr=[F_elo,F_ehi],
                 error_kw=dict(elinewidth=1.4,capsize=5,capthick=1.4,ecolor='#222'),
                 zorder=3,edgecolor='white')
    for bar,h in zip(bars,hatches): bar.set_hatch(h)
    ax1.axhline(1.0,color=C_IDEAL,ls='--',lw=1.1,alpha=0.6)
    ax1.set_xticks(x); ax1.set_xticklabels(labels,fontsize=9)
    ymin1=max(0.5,min(F_vals)-0.06)
    ax1.set_ylim(ymin1,1.06); ax1.set_ylabel('$F(\\rho_Y,\\rho_M)$')
    ax1.set_title('(a) Output fidelity',fontsize=11)
    ax1.yaxis.grid(True,alpha=0.3,zorder=0)
    for xi,fv in enumerate(F_vals):
        ax1.text(xi,fv+F_ehi[xi]+0.004,f'{fv:.4f}',ha='center',fontsize=8.5)
    um_F=res['unmitigated']['F']
    for xi,(k,fv) in enumerate(zip(keys[1:],F_vals[1:]),start=1):
        d=fv-um_F; s='+' if d>=0 else ''
        col=C_IDEAL if d>0 else '#E74C3C'
        ax1.text(xi,ymin1+0.01,f'{s}{d:.4f}',ha='center',fontsize=8,
                 color=col,style='italic')

    # ── (b) ZNE extrapolation ─────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1])
    zne=res['zne']; lams=zne['lambdas']
    Fz=[zne['F_at_lambda'][str(l)] for l in lams]
    lam_ext=np.linspace(0,max(lams)*1.05,200)
    ax2.scatter(lams,Fz,color='#C0392B',s=65,zorder=5)
    ax2.plot(lam_ext,np.polyval(zne['fit_coeffs'],lam_ext),
             color='#C0392B',lw=2.0,ls='--',
             label=f'Linear fit → $F_{{\\rm ZNE}}={zne["F"]:.4f}$')
    ax2.scatter([0],[zne['F']],color='#C0392B',s=110,marker='*',zorder=6)
    ax2.axhspan(zne['F_lo'],zne['F_hi'],alpha=0.15,color='#C0392B')
    ax2.axhline(res['unmitigated']['F'],color='#7F8C8D',ls=':',lw=1.3,alpha=0.8,
                label=f'Unmitigated={res["unmitigated"]["F"]:.4f}')
    ax2.axhline(1.0,color=C_IDEAL,ls='-.',lw=1.1,alpha=0.6,label='Ideal')
    ax2.set_xlabel('$\\lambda$ (noise factor)')
    ax2.set_ylabel('$F(\\rho_Y,\\rho_M)$')
    ax2.set_title('(b) ZNE extrapolation',fontsize=11)
    ax2.legend(fontsize=8.5,framealpha=0.9)
    ax2.set_xlim(-0.3,max(lams)+0.5)
    ax2.yaxis.grid(True,alpha=0.3,zorder=0)

    # ── (c) Bootstrap overlay ────────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[2])
    for key,label,color,_ in CONDITIONS:
        r=res.get(key)
        if r is None: continue
        bsF=np.array(r['bootstrap_F']); med=np.median(bsF)
        lo=np.percentile(bsF,2.5); hi=np.percentile(bsF,97.5)
        ax3.hist(bsF,bins=40,color=color,alpha=0.60,
                 edgecolor='white',lw=0.4,zorder=3,
                 label=f'{label}  $F={r["F"]:.4f}$')
        ax3.axvline(med,color=color,lw=1.7,zorder=4)
        ax3.axvline(lo, color=color,lw=0.9,ls='--',zorder=4)
        ax3.axvline(hi, color=color,lw=0.9,ls='--',zorder=4)
    ax3.axvline(1.0,color=C_IDEAL,lw=1.5,ls=':',zorder=5,label='Ideal')
    ax3.set_xlabel('$F(\\rho_Y,\\rho_M)$'); ax3.set_ylabel('Count')
    ax3.set_title('(c) Bootstrap distributions',fontsize=11)
    ax3.legend(framealpha=0.9,edgecolor='#ccc',fontsize=8,loc='upper left')
    ax3.yaxis.grid(True,alpha=0.3,zorder=0)

    fig.suptitle(
        f'Error mitigation baseline — ibm\\_torino  ({shots:,} shots/basis)\n'
        f'DD (XX seq.) · Readout (mthree) · ZNE ($\\lambda\\in${{{lams}}}, linear)',
        fontsize=11, y=1.02
    )
    fig.tight_layout()
    return fig

# =============================================================================
#  Save + print summary
# =============================================================================
print("Generating error mitigation figures ...")
for fname, fn in [
    ('fig_em1_bar_comparison.pdf',  fig_bar),
    ('fig_em2_zne_extrapolation.pdf', fig_zne),
    ('fig_em3_bootstrap_overlay.pdf', fig_bootstrap),
    ('fig_em_panel.pdf',            fig_panel),
]:
    f = fn()
    f.savefig(os.path.join(OUT_DIR, fname), bbox_inches='tight', dpi=200)
    plt.close(f)
    print(f"  [✓] {fname}")

print(f"\n{'='*60}")
print("  PAPER-READY NUMBERS")
print(f"{'='*60}")
um = res['unmitigated']
print(f"  Backend     : {meta['backend']}  ({shots:,} shots/basis)")
print(f"  DD sequence : {meta['dd_sequence']}")
print(f"  ZNE factors : {meta['zne_factors']}\n")
print(f"  {'Condition':22s}  {'F':>8}  {'95% CI':>22}  {'ΔF':>8}  {'p_succ':>8}")
print("  " + "-"*72)
for key, label, _, _ in CONDITIONS:
    r = res.get(key)
    if r is None: continue
    delta = r['F'] - um['F']
    s = '+' if delta >= 0 else ''
    ci = f"[{r['F_lo']:.4f},{r['F_hi']:.4f}]"
    print(f"  {label:22s}  {r['F']:>8.4f}  {ci:>22}  "
          f"{s}{delta:>7.4f}  {r['p_succ']:>8.4f}")
print(f"\n[✓] Done.")

Generating error mitigation figures ...
  [✓] fig_em1_bar_comparison.pdf
  [✓] fig_em2_zne_extrapolation.pdf
  [✓] fig_em3_bootstrap_overlay.pdf


/var/folders/dg/bj9b922n29vcxp68247bp7vh0000gn/T/ipykernel_7961/2957474675.py:307: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


  [✓] fig_em_panel.pdf

  PAPER-READY NUMBERS
  Backend     : ibm_torino  (10,000 shots/basis)
  DD sequence : XX
  ZNE factors : [1, 3, 5]

  Condition                      F                  95% CI        ΔF    p_succ
  ------------------------------------------------------------------------
  Unmitigated               0.8395         [0.8223,0.8568]  + 0.0000    0.2354
  DD (XX)                   0.7522         [0.7346,0.7703]  -0.0872    0.2409
  ZNE
($\lambda$=1,3,5)     0.8420         [0.8217,0.8629]  + 0.0025    0.2377

[✓] Done.
